# Exercise — Encode Governance Obligations in a Data Contract

**Trailhead Provisions** has an ungoverned `orders` contract: it describes columns but carries no
policy. Extend it with a **governance block** so the validator passes, then explain how a policy
change propagates across producers and consumers. See `INSTRUCTIONS.md`.

In [ ]:
from governance_toolkit import validate_contract

contract = {
    "name": "orders",
    "domain": "E-commerce Orders",
    "owner": "orders-team@trailhead.example",
    "version": "1.0.0",
    "columns": [
        {"name": "order_id", "type": "integer"},
        {"name": "customer_id", "type": "integer"},
        {"name": "order_ts", "type": "timestamp"},
        {"name": "status", "type": "string"},
        {"name": "amount", "type": "double"},
        {"name": "currency", "type": "string"},
    ],
}
ok, errs = validate_contract(contract)
print("Base contract valid?", ok, "| errors:", errs)   # provided: fails, no governance block

## 1. Add the governance block
Add `pii_tags`, `retention` (e.g. `P7Y`), and `quality_slos` (list of `{dimension, threshold}`), and mark PII columns with `col["pii"] = True`. Make `validate_contract` pass.

In [ ]:
for col in contract["columns"]:
    col["pii"] = col["name"] in ("customer_id",)   # FK to a PII entity
contract["governance"] = {
    "pii_tags": ["customer_id"],
    "retention": "P7Y",
    "quality_slos": [
        {"dimension": "completeness", "threshold": 0.99},
        {"dimension": "validity", "threshold": 0.98},
        {"dimension": "accuracy", "threshold": 0.995},
    ],
}
ok, errs = validate_contract(contract)
print("Extended contract valid?", ok, "| errors:", errs)
assert ok, "fix the governance block until the validator passes"
contract["governance"]

## 2. Policy-propagation write-up
Replace the cell below with your write-up.

### Policy-propagation write-up

Adding `retention: P7Y` and PII tags to the `orders` contract is not a local change.

- **Producers** (the Orders pipeline) must now emit records that meet the quality SLOs and
  enforce a 7-year retention/deletion lifecycle — a code and storage change.
- **Consumers** that join `orders.customer_id` to customer data inherit the PII obligation: the
  join result is PII even though `orders` alone looks benign, so downstream products
  (`customer_360`, `marketing_audience`) must re-classify and re-tag.
- **Propagation mechanism:** the change **version-bumps** the contract (1.0.0 → 1.1.0). Every
  downstream contract that depends on `orders` is flagged for review; the central schema gate
  refuses to publish a downstream product referencing an outdated upstream version. Remediation
  is upstream-first: fix `orders`, re-validate, then walk the lineage downstream in topological
  order.